# Notebook 02: Data Cleaning
Remove cancellations, null CustomerIDs, invalid Quantity/UnitPrice; add Revenue and YearMonth.

**Input:** `data/raw/Online Retail.xlsx`  
**Output:** `data/preprocessed/online_retail_preprocessed.csv`

In [1]:
import os
import sys
import pandas as pd

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, '.')

from src.data_cleaning import run_cleaning_pipeline, load_raw

# Before: raw data profile
raw_path = 'data/raw/Online Retail.xlsx'
df_raw = load_raw(raw_path)
print(f"RAW: {len(df_raw):,} rows, {df_raw['CustomerID'].notna().sum():,} with CustomerID")
print(df_raw[['Quantity', 'UnitPrice', 'CustomerID']].describe(include='all'))

# Run full cleaning pipeline


df_cleaned = run_cleaning_pipeline(
    data_path=raw_path,
    save_path='data/preprocessed/online_retail_preprocessed.csv'
)

print('\nCLEANED SUMMARY:')
print(f"Rows: {len(df_cleaned):,}")
print(f"Unique customers: {df_cleaned['CustomerID'].nunique():,}")
print(f"Unique invoices: {df_cleaned['InvoiceNo'].nunique():,}")
print('\nNull checks (should be 0 for CustomerID, negative Quantity/UnitPrice removed):')
print(df_cleaned[['CustomerID']].isna().sum())

df_cleaned.head()

RAW: 541,909 rows, 406,829 with CustomerID
            Quantity      UnitPrice     CustomerID
count  541909.000000  541909.000000  406829.000000
mean        9.552250       4.611114   15287.690570
std       218.081158      96.759853    1713.600303
min    -80995.000000  -11062.060000   12346.000000
25%         1.000000       1.250000   13953.000000
50%         3.000000       2.080000   15152.000000
75%        10.000000       4.130000   16791.000000
max     80995.000000   38970.000000   18287.000000


Started with: 541,909 rows, 406,829 with CustomerID


  Identified 9,288 cancellation rows (C-prefix) and 3 accounting adjustment rows (A-prefix)
After removing cancellations/adjustments: 532,618 rows
After Quantity/UnitPrice/CustomerID filters: 397,884 rows


  Found 5,192 duplicate rows across 1884 invoices
  Dropped 5,192 duplicate rows

  OUTLIER INVESTIGATION: 3 rows with Revenue > £10,000
  These represent £284,623 (3.2% of total revenue)
    Customer 12346: MEDIUM CERAMIC TOP STORAGE JAR | Qty=74,215 | Revenue=£77,184
    Customer 15098: PICNIC BASKET WICKER 60 PIECES | Qty=60 | Revenue=£38,970
    Customer 16446: PAPER CRAFT , LITTLE BIRDIE | Qty=80,995 | Revenue=£168,470
  Decision: Retained in cleaned data (revenue reporting). Downstream RFM/clustering should cap at 99th percentile.

Final preprocessed rows: 392,692
Unique customers: 4,338
Unique invoices: 18,532



CLEANED SUMMARY:
Rows: 392,692
Unique customers: 4,338
Unique invoices: 18,532

Null checks (should be 0 for CustomerID, negative Quantity/UnitPrice removed):
CustomerID    0
dtype: int64


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Revenue,YearMonth,is_non_product
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom,15.30,2010-12,False
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12,False
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom,22.00,2010-12,False
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12,False
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom,20.34,2010-12,False
